# Alternativa evaluada: Random Forest vs. Red bayesiana
### Sistema de Recomendación Vocacional · Proyecto MIA (UDLA)

El proyecto eligió una **red bayesiana**. Este notebook construye la alternativa más común —un **Random Forest**
(aprendizaje automático supervisado)— con **las mismas entradas** (notas + CHASIDE) y **las mismas salidas**
(interés y rendimiento por área, zona del plano y carreras), y compara los resultados con el escenario de evaluación
de la tesis (Módulo 3B: 500 estudiantes, CHASIDE simulado con verdad conocida, 70 % entrenamiento / 30 % prueba).

| | Red bayesiana (proyecto) | Random Forest (alternativa) |
|---|---|---|
| Qué es | Modelo probabilístico: combina fuentes según su confiabilidad (prior + evidencia → posterior) | Conjunto de árboles de decisión que aprende reglas a partir de ejemplos |
| Qué necesita para aprender | Solo notas y CHASIDE (**sin etiquetas**) + conocimiento experto (rúbrica de confiabilidad) | Ejemplos con la **respuesta correcta** (interés y rendimiento "verdaderos") |
| Incertidumbre | Exacta: cada valor trae su rango y la probabilidad de cada zona | No nativa: se aproxima con la dispersión entre árboles |
| Explicación por estudiante | Sí: cuánto aportó cada fuente a cada área | No: solo importancia global de variables |

> **Ventaja deliberada para la alternativa:** al Random Forest se le entregan los valores verdaderos de la simulación
> para entrenar. En el colegio **esos valores no existen** (nadie conoce el "interés verdadero" de un estudiante), así
> que en la práctica ni siquiera podría entrenarse sin años de seguimiento de egresados.

## 0 · Preparación

In [ ]:
import os, sys, json
BASE = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'alternativa' else os.path.abspath('.')
sys.path.insert(0, BASE); sys.path.insert(0, os.path.join(BASE, 'alternativa'))
import numpy as np, pandas as pd
import comparacion as cmp
from modelo import bayes, configuracion as cfg
claves = cfg.cargar_claves(); fuentes = cfg.cargar_fuentes(); lista_car, _ = cfg.cargar_carreras()
calib_car = json.load(open(f'{BASE}/config/calibracion_referencia.json', encoding='utf-8'))['carreras']
print('Listo ·', len(claves), 'ítems CHASIDE ·', len(lista_car), 'carreras')

## 1 · Escenario de evaluación (el mismo de la tesis)
Rendimiento real de los 500 estudiantes (Módulo 3) + CHASIDE simulado con el generador **realista** del Módulo 3B
(ρ interés–rendimiento 0.2, autopercepción λ 0.4, sesgo personal 0.3, 3 % de respuestas al azar).

In [ ]:
rend, por_anio = cmp.cargar_notas()
apps, verdad, aleatorio = cmp.simular(rend, por_anio, claves, 'realista')
EV, CRIT, cal = cmp.evidencia(rend, apps, claves)
train, test = map(sorted, bayes.particion(EV.keys()))
print(f'{len(EV)} estudiantes · entrenamiento {len(train)} · prueba {len(test)}')

## 2 · Los dos modelos
- **Red bayesiana:** aprende α, λ y ρ de los datos **sin mirar la verdad** y calcula el posterior exacto (Módulo 5).
- **Random Forest:** dos bosques de 300 árboles (interés y rendimiento) + un clasificador de zona, entrenados **con la verdad**.
  Variables: z del CHASIDE (interés y aptitud), z de notas, evidencia W, criterios de calidad y el área.

In [ ]:
P = cmp.bayes_ajustar(EV, por_anio, train, cal)
pred_bayes = cmp.bayes_predecir(EV, CRIT, P, cal, test, fuentes)
modelos_rf = cmp.rf_ajustar(EV, CRIT, verdad, train)
pred_rf = cmp.rf_predecir(modelos_rf, EV, CRIT, test)
print('Predicciones:', len(pred_bayes), 'filas estudiante×área por modelo')

## 3 · Resultados en el conjunto de prueba

In [ ]:
NOMBRES = {'r_I': 'Correlación con el interés real', 'r_R': 'Correlación con el rendimiento real',
           'RMSE_I': 'Error del interés (RMSE)', 'RMSE_R': 'Error del rendimiento (RMSE)',
           'cobertura90_I': 'Rango 90 % contiene el interés real', 'cobertura90_R': 'Rango 90 % contiene el rendimiento real',
           'exactitud_zona': 'Zona correcta (cuando asigna zona)', 'indeterminados': 'Áreas que deja indeterminadas',
           'exactitud_zona_todas': 'Zona correcta (todas las áreas)', 'brier': 'Error de Brier de las zonas (menor = mejor)',
           'top1_en_top3': 'Área de mayor interés dentro de su top 3', 'top5_en_comun': 'Carreras del top 5 que coinciden con el real (de 5)',
           'carrera1_real_en_top5': 'Carrera #1 real dentro del top 5'}
mb = {**cmp.metricas(pred_bayes, verdad), **cmp.ranking_carreras(pred_bayes, verdad, lista_car, calib_car)}
mr = {**cmp.metricas(pred_rf, verdad), **cmp.ranking_carreras(pred_rf, verdad, lista_car, calib_car)}
tabla = pd.DataFrame({'Red bayesiana': mb, 'Random Forest': mr}).rename(index=NOMBRES).round(3)
tabla

**Cómo leerlo.** En precisión de las estimaciones los dos son prácticamente iguales (la red bayesiana incluso un poco
mejor), **a pesar de que el Random Forest vio la respuesta correcta**. La diferencia grande está en la **honestidad
de la incertidumbre**: cuando el Random Forest dice "rango del 90 %", el valor real cae dentro solo ~68 % de las veces
(es sobreconfiado); la red bayesiana cumple lo que promete (~88–91 %). Para orientar a un adolescente, saber *cuándo*
el sistema no está seguro es tan importante como el resultado.

## 4 · ¿Y con pocos estudiantes? (curva de aprendizaje)

In [ ]:
rng = np.random.default_rng(1); filas = []
for n in (20, 50, 100, 200, 350):
    for rep in range(5 if n < 350 else 1):
        sub = sorted(rng.choice(train, size=n, replace=False)) if n < len(train) else train
        b = cmp.metricas(cmp.bayes_predecir(EV, CRIT, cmp.bayes_ajustar(EV, por_anio, sub, cal), cal, test, fuentes), verdad)
        r = cmp.metricas(cmp.rf_predecir(cmp.rf_ajustar(EV, CRIT, verdad, sub, semilla=rep), EV, CRIT, test), verdad)
        filas += [{'n': n, 'modelo': 'Red bayesiana', **b}, {'n': n, 'modelo': 'Random Forest', **r}]
curva = pd.DataFrame(filas).groupby(['n', 'modelo'])[['r_I', 'r_R', 'cobertura90_I', 'brier']].mean().round(3).unstack()
curva

La red bayesiana rinde igual con 20 estudiantes que con 350, porque su estructura viene del conocimiento experto
(qué fuente mide qué y con cuánta confiabilidad). El Random Forest necesita muchos ejemplos etiquetados y aun con
350 no alcanza la calidad probabilística de la red bayesiana.

In [ ]:
from IPython.display import Image, display
import subprocess
subprocess.run([sys.executable, os.path.join(BASE, 'alternativa', 'graficos.py')], check=False)
for f in ('fig1_confianza.png', 'fig2_pocos_datos.png'):
    p = os.path.join(BASE, 'alternativa', 'resultados', f)
    if os.path.exists(p): display(Image(p, width=640))

## 5 · El mismo estudiante con los dos modelos

In [ ]:
def comparar_estudiante(id_est):
    b = pred_bayes[pred_bayes.id == id_est].set_index('area'); r = pred_rf[pred_rf.id == id_est].set_index('area')
    pct = lambda z: round(float(100 * __import__('scipy').stats.norm.cdf(z)))
    filas = []
    for a in cmp.CODIGOS:
        iv, rv = verdad[(id_est, a)]
        filas.append({'área': a,
                      'real: interés P / rend. P / zona': f"{pct(iv)} / {pct(rv)} / {cmp.zona_de(iv, rv)}",
                      'bayes: interés P (rango 90 %)': f"{pct(b.loc[a,'mu_I'])} ({pct(b.loc[a,'mu_I']-1.645*b.loc[a,'sd_I'])}–{pct(b.loc[a,'mu_I']+1.645*b.loc[a,'sd_I'])})",
                      'bayes: zona': b.loc[a, 'zona'],
                      'RF: interés P (rango 90 %)': f"{pct(r.loc[a,'mu_I'])} ({pct(r.loc[a,'lo_I'])}–{pct(r.loc[a,'hi_I'])})",
                      'RF: zona': r.loc[a, 'zona']})
    return pd.DataFrame(filas).set_index('área')

ejemplo = test[0]
print('Estudiante', ejemplo, '(respondió al azar)' if aleatorio[ejemplo] else '')
comparar_estudiante(ejemplo)

## 6 · Conclusión
1. **Precisión:** empate técnico; la red bayesiana iguala o supera al Random Forest **sin necesitar la respuesta correcta**.
2. **Incertidumbre honesta:** la red bayesiana cumple su 90 %; el Random Forest promete 90 % y acierta ~68 %.
3. **Pocos datos:** la red bayesiana es estable desde 20 estudiantes; el Random Forest se degrada.
4. **Viabilidad real:** sin etiquetas (que el colegio no tiene) el Random Forest no puede entrenarse.
5. **Explicabilidad:** la red bayesiana explica cada resultado por fuente (notas, intereses, autopercepción); el bosque no.

**Por eso el proyecto usa la red bayesiana.** El Random Forest podría servir en el futuro como *complemento*, si el
colegio acumula datos de seguimiento de egresados (carrera elegida y satisfacción) para validar las recomendaciones.